# Chapter 7: Process Modeling with LLMs

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Process Discovery Engine** that takes natural language process descriptions and generates structured process models, swimlane diagrams (as Mermaid code), and identifies optimization opportunities.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Basic understanding of BPMN and process modeling


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Natural Language to Process Model

Convert a text description of a business process into a structured process model.


In [ ]:
# Natural language process description
process_description = """
Employee Onboarding Process:

When a new hire accepts an offer, HR creates their employee record in the system.
Then IT provisions their laptop, email account, and system access -- these three 
can happen in parallel. Meanwhile, HR schedules orientation and assigns a buddy.

Once IT is done with provisioning, the manager is notified to prepare a 30-60-90 
day plan. The new hire completes mandatory compliance training within the first week.

If the compliance training is not completed within 5 business days, an escalation 
email goes to the manager and HR. If completed, the employee gets access to 
project-specific systems.

After the first week, the buddy checks in. After 30 days, the manager conducts 
a check-in review. The onboarding is considered complete when the 30-day review 
is done and all training is passed.
"""

# Extract structured process model
process_prompt = f"""Analyze this business process and extract a structured process model.

Process Description:
{process_description}

Return a JSON object with:
- process_name: name of the process
- actors: list of roles involved
- steps: array of {{id, name, actor, type (task/decision/parallel_gateway/event), 
  description, inputs, outputs, next_steps (list of step IDs)}}
- start_event: step ID
- end_event: step ID  
- sla_requirements: any time constraints mentioned
- exceptions: error/escalation paths

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': process_prompt}],
    temperature=0.2
)
process_model = json.loads(response.choices[0].message.content)
print(f"Process: {process_model['process_name']}")
print(f"Actors: {', '.join(process_model['actors'])}")
print(f"Steps: {len(process_model['steps'])}")
for step in process_model['steps']:
    print(f"  [{step['type']}] {step['id']}: {step['name']} ({step['actor']})")

## 2. Generate Mermaid Diagram

Convert the process model to a Mermaid flowchart that can be rendered in any Markdown viewer.


In [ ]:
# Generate Mermaid diagram from the process model
mermaid_prompt = f"""Convert this process model into a Mermaid flowchart diagram.

Process Model: {json.dumps(process_model)}

Requirements:
- Use 'graph TD' (top-down) layout
- Use different shapes: rectangles for tasks, diamonds for decisions, 
  rounded rectangles for start/end events
- Label edges with conditions where applicable
- Group by actor using subgraph blocks
- Add styling for different step types

Return ONLY the Mermaid code, no markdown fences."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': mermaid_prompt}],
    temperature=0.2
)
mermaid_code = response.choices[0].message.content
print('Generated Mermaid diagram:\n')
print(mermaid_code)
print('\n--- Copy the above into a Mermaid-compatible renderer to visualize ---')

In [ ]:
# Process optimization analysis
optimize_prompt = f"""As a process improvement consultant, analyze this business process 
for optimization opportunities.

Process Model: {json.dumps(process_model)}

Identify:
1. Bottlenecks: steps that could delay the overall process
2. Automation candidates: tasks that could be fully or partially automated
3. Parallel opportunities: sequential tasks that could run in parallel
4. Redundancies: unnecessary or duplicate steps
5. Risk points: where the process is most likely to fail

For each finding, provide:
- category: one of the 5 above
- affected_steps: list of step IDs
- description: what the issue is
- recommendation: how to improve
- estimated_impact: time savings or quality improvement

Return as a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': optimize_prompt}],
    temperature=0.3
)
optimizations = json.loads(response.choices[0].message.content)
print(f'Found {len(optimizations)} optimization opportunities:\n')
for opt in optimizations:
    print(f"[{opt['category'].upper()}] {opt['description']}")
    print(f"  Recommendation: {opt['recommendation']}")
    print(f"  Impact: {opt['estimated_impact']}")
    print()

## Exercises
1. Model a process from your own organization and generate the Mermaid diagram.
2. Add a step that generates RACI matrix from the process model.
3. Create a process comparison tool that identifies differences between current-state and future-state.
4. Generate BPMN 2.0 XML output instead of Mermaid -- this can be imported into tools like Camunda.
